In [2]:
from __future__ import annotations
from geopy.geocoders import Nominatim
from dataclasses import dataclass

@dataclass
class Location:
    name: str
    latitude: float
    longitude: float

_geolocator = None

def geocode_city(city: str, user_agent: str = "pogoda-app") -> Location:
    """Geocode a city name into coordinates using Nominatim.

    Parameters
    ----------
    city: str
        City name (can include country, e.g., "Szczecin, Poland").
    user_agent: str
        Required by Nominatim usage policy.

    Returns
    -------
    Location

    Raises
    ------
    ValueError: if no result is found.
    """
    global _geolocator
    if _geolocator is None:
        _geolocator = Nominatim(user_agent=user_agent, timeout=10)
    result = _geolocator.geocode(city)
    if not result:
        raise ValueError(f"City not found: {city}")23262
    return Location(name=result.address, latitude=result.latitude, longitude=result.longitude)


In [25]:
import xarray as xr

# Open dataset
ds = xr.open_dataset("tg_ens_mean_0.25deg_reg_2011-2024_v31.0e.nc")

# Work only with 'tg' and drop missing values
#df = ds["tg"].to_dataframe(name="tg").reset_index()

# Keep only rows where tg is not NaN
#df = df.dropna(subset=["tg"])

# Save to CSV
#df.to_csv("output_nonmissing.csv", index=False)
#print("CSV with non-empty tg values saved!")

In [3]:
cairo=geocode_city("Cairo")

In [26]:
n_lat = ds.dims["latitude"]
n_lon = ds.dims["longitude"]

# Total distinct grid cells
n_locations = n_lat * n_lon

print(f"Lat points: {n_lat}, Lon points: {n_lon}, Distinct locations: {n_locations}")


Lat points: 201, Lon points: 464, Distinct locations: 93264


C:\Users\mmachura\AppData\Local\Temp\ipykernel_13120\1049206283.py:1: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  n_lat = ds.dims["latitude"]
C:\Users\mmachura\AppData\Local\Temp\ipykernel_13120\1049206283.py:2: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  n_lon = ds.dims["longitude"]


In [4]:
ts = ds["tg"].sel(latitude=cairo.latitude, longitude=cairo.longitude, method="nearest").to_series()

In [27]:
mask_valid = ds["tg"].notnull().any(dim="time")

# Count how many grid cells are valid
n_valid_locations = mask_valid.sum().item()

In [29]:
n_valid_locations*12*50/1000000


12.6288

In [6]:
# If ts is still an xarray.DataArray:
ts_clean = ts   # convert to pandas Series (index = time)

# Count NaNs per year
nan_per_year = ts_clean.isna().groupby(ts_clean.index.year).sum()

print(nan_per_year)

time
2011      0
2012      0
2013      0
2014      0
2015      0
2016      0
2017     57
2018     34
2019      2
2020     62
2021    365
2022    365
2023    365
2024    366
Name: tg, dtype: int64


In [ ]:
ts[["time"=="2016"]]

In [22]:
box = ds["tg"].sel(
    latitude=slice(cairo.latitude-0.2, cairo.latitude+0.2),
    longitude=slice(cairo.longitude-0.2, cairo.longitude+0.2)
).dropna(dim="time")

In [9]:
box

<xarray.DataArray 'tg' (time: 5114, latitude: 4, longitude: 4)> Size: 327kB
[81824 values with dtype=float32]
Coordinates:
  * longitude  (longitude) float64 32B 30.88 31.12 31.38 31.62
  * latitude   (latitude) float64 32B 29.62 29.88 30.12 30.38
  * time       (time) datetime64[ns] 41kB 2011-01-01 2011-01-02 ... 2024-12-31
Attributes:
    units:          Celsius
    long_name:      mean temperature
    standard_name:  air_temperature

In [10]:
yearly_counts = box.groupby("time.year").count(dim="time")
yearly_missing = box.groupby("time.year").apply(lambda x: x.isnull().sum(dim="time"))

In [14]:
# Non-missing values per (lat, lon)
rowcount = box.count(dim="time")   # DataArray with dims (latitude, longitude)

# Missing values per (lat, lon)
nancount = box.isnull().sum(dim="time")  # same dims

In [17]:
df_counts = (
    xr.Dataset({"rowcount": rowcount, "nancount": nancount})
      .to_dataframe()
      .reset_index()
)

print(df_counts.head(10))

   longitude  latitude  rowcount  nancount
0     31.125    29.875      3498      1616
1     31.125    30.125      3498      1616
2     31.375    29.875      3498      1616
3     31.375    30.125      3498      1616


In [23]:
import xarray as xr
import pandas as pd
import numpy as np

# 1) Select your spatial box
box = ds["tg"].sel(
    latitude=slice(cairo.latitude - 0.5, cairo.latitude + 0.5),
    longitude=slice(cairo.longitude - 0.5, cairo.longitude + 0.5),
)

# 2) Monthly aggregation across *time AND space*
# Monthly mean over all dims (NaNs ignored)
monthly_mean = box.resample(time="1MS").mean(dim=("time", "latitude", "longitude"))

# Count of non-NaN values contributing each month across all dims
nonmissing = box.resample(time="1MS").count(dim=("time", "latitude", "longitude"))

# Total expected values = (#days in month) * (#grid cells in box)
n_cells = box.sizes["latitude"] * box.sizes["longitude"]
days_per_month = box["time"].resample(time="1MS").count()            # time-only
total_vals = days_per_month * n_cells                                 # time-only

# NaN count per month
missing = total_vals - nonmissing

# 3) Assemble tidy DataFrame: one row per month for the whole box
df_monthly_box = pd.DataFrame({
    "time": monthly_mean["time"].values,
    "tg_mean": monthly_mean.values,
    "count_nonmissing": nonmissing.values.astype(np.int64),
    "count_missing": missing.values.astype(np.int64),
})
df_monthly_box["year"] = pd.to_datetime(df_monthly_box["time"]).dt.year
df_monthly_box["month"] = pd.to_datetime(df_monthly_box["time"]).dt.month
df_monthly_box = df_monthly_box[["year","month","tg_mean","count_nonmissing","count_missing"]]

print(df_monthly_box.head(120))
# df_monthly_box.to_csv("tg_box_monthly_mean.csv", index=False)

     year  month    tg_mean  count_nonmissing  count_missing
0    2011      1  16.296774               496              0
1    2011      2  16.901831               448              0
2    2011      3  17.976595               496              0
3    2011      4  20.998251               480              0
4    2011      5  23.941732               496              0
..    ...    ...        ...               ...            ...
115  2020      8  30.339556               496              0
116  2020      9  30.493019               480              0
117  2020     10  26.926128               496              0
118  2020     11        NaN                 0            480
119  2020     12        NaN                 0            496

[120 rows x 5 columns]


In [1]:
l = [1,3,2]

In [2]:
l

[1, 3, 2]

In [3]:
l.sort()

In [4]:
l

[1, 2, 3]

In [5]:
3^3

0

In [6]:
3^3^3

3

In [12]:
import math

In [9]:
data = [1,2,3,float("nan")]

True